# NovaMart Sales Analytics - EDA & Cleaning

This notebook loads retail sales data, performs cleaning, and prepares a dataset for dashboarding in Tableau/Power BI.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
ROOT = Path('..')
raw_file = ROOT / 'data' / 'raw' / 'sales_data.csv'
processed_file = ROOT / 'data' / 'processed' / 'sales_clean.csv'
raw_file

## Load data


In [ ]:
df = pd.read_csv(raw_file)
df.head()

## Data cleaning
- Standardize column names
- Parse order date
- Handle missing discounts
- Remove duplicate order rows
- Recompute sales/profit metrics


In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['discount'] = df.get('discount', 0).fillna(0).clip(lower=0, upper=1)
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce').fillna(0)
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce').fillna(0)
if 'unit_cost' not in df.columns:
    df['unit_cost'] = np.nan
df['unit_cost'] = pd.to_numeric(df['unit_cost'], errors='coerce').fillna(df['unit_price'] * 0.6)
df = df.drop_duplicates(subset=['order_id', 'product_id'])
df = df[df['order_date'].notna()]
df['gross_sales'] = df['quantity'] * df['unit_price']
df['net_sales'] = df['gross_sales'] * (1 - df['discount'])
df['total_cost'] = df['quantity'] * df['unit_cost']
df['profit'] = df['net_sales'] - df['total_cost']
df['order_month'] = df['order_date'].dt.to_period('M').astype(str)
df['order_quarter'] = df['order_date'].dt.to_period('Q').astype(str)
df.head()

## Exploratory analysis


In [ ]:
region_summary = (
    df.groupby('region', as_index=False)
      .agg(total_revenue=('net_sales', 'sum'), total_profit=('profit', 'sum'), units=('quantity', 'sum'))
      .sort_values('total_revenue', ascending=False)
)
top_products = (
    df.groupby(['product_id', 'product_name'], as_index=False)
      .agg(revenue=('net_sales', 'sum'), profit=('profit', 'sum'), units=('quantity', 'sum'))
      .sort_values('revenue', ascending=False)
      .head(10)
)
region_summary, top_products

## Export cleaned dataset


In [ ]:
processed_file.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(processed_file, index=False)
processed_file